# 01 · Data Exploration & Preprocessing

**Owner: Member 1.**

Deep-dive companion to §2 of `main.ipynb`. We dig into the
statistical structure of the UCI Heart Disease (Cleveland)
dataset and validate the clinical bin choices used in
`src/preprocessing.py`.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
from src.data_loader import load_heart_disease, describe_schema
df_raw = load_heart_disease()
print(df_raw.shape)
df_raw.describe(include='all').T.head(15)

## Continuous variable distributions

Are the clinical bin edges (JNC-7 / ATP-III etc.) reasonable
given *this* sample? We overlay the chosen cut points on the
empirical KDE.

In [ ]:
from src.preprocessing import DISCRETIZATION
cont_vars = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, var in zip(axes.flat, cont_vars):
    sns.kdeplot(df_raw[var].dropna(), fill=True, ax=ax)
    edges, _ = DISCRETIZATION[var]
    for e in edges[1:-1]:
        ax.axvline(e, color='crimson', linestyle='--', linewidth=1)
    ax.set_title(var)
for ax in axes.flat[len(cont_vars):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## Stratified by outcome

How separable are the continuous variables once we condition
on disease status?

In [ ]:
df_raw_bin = df_raw.copy()
df_raw_bin['target'] = (df_raw_bin['num'] > 0).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, v in zip(axes, ['age', 'chol', 'thalach']):
    sns.kdeplot(data=df_raw_bin, x=v, hue='target', fill=True, common_norm=False, ax=ax)
    ax.set_title(v)
plt.tight_layout(); plt.show()

## Correlation vs. mutual information

A correlation matrix can miss non-monotonic dependence. We
contrast both views.

In [ ]:
from src.preprocessing import PreprocessConfig, build_dataset
from src.eda import mutual_information_matrix, plot_mi_heatmap

df = build_dataset(df_raw, PreprocessConfig())
mi = mutual_information_matrix(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
df_num = df_raw.replace('?', np.nan).apply(pd.to_numeric, errors='coerce')
sns.heatmap(df_num.corr().abs(), ax=axes[0], cmap='rocket_r', annot=True, fmt='.2f',
            cbar_kws={'label': '|Pearson r|'})
axes[0].set_title('|Pearson correlation| (raw)')
sns.heatmap(mi, ax=axes[1], cmap='rocket_r', annot=True, fmt='.2f',
            cbar_kws={'label': 'MI (nats)'})
axes[1].set_title('Mutual information (discretized)')
plt.tight_layout(); plt.show()

## χ² test of independence with the target

In [ ]:
from src.eda import chi_square_table
chi_square_table(df)